In [109]:
#imports

!pip install numpy pandas matplotlib scikit-learn ucimlrepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from ucimlrepo import fetch_ucirepo
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [110]:
#loading the dataset
student_performance = fetch_ucirepo(id=320)
X = student_performance.data.features
y = student_performance.data.targets


In [111]:
#inspecting the dataset
print(f"Features number of rows :{X.shape[0]}") #number of rows in the features dataset
print(f"Targets number of rows :{y.shape[0]}") #number of rows in the targets dataset
print(f"Features number of columns :{X.shape[1]}") #number of columns in the features dataset
print(f"Targets number of columns :{y.shape[1]}") #number of columns in the targets dataset

Features number of rows :649
Targets number of rows :649
Features number of columns :30
Targets number of columns :3


- Number of rows match up, shows indexing is done well

In [112]:
#inspecting variable types
X.info()
y.info()
#most of the features are categorical and the target is numerical, whilst some of the features are numerical

<class 'pandas.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 30 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      649 non-null    str  
 1   sex         649 non-null    str  
 2   age         649 non-null    int64
 3   address     649 non-null    str  
 4   famsize     649 non-null    str  
 5   Pstatus     649 non-null    str  
 6   Medu        649 non-null    int64
 7   Fedu        649 non-null    int64
 8   Mjob        649 non-null    str  
 9   Fjob        649 non-null    str  
 10  reason      649 non-null    str  
 11  guardian    649 non-null    str  
 12  traveltime  649 non-null    int64
 13  studytime   649 non-null    int64
 14  failures    649 non-null    int64
 15  schoolsup   649 non-null    str  
 16  famsup      649 non-null    str  
 17  paid        649 non-null    str  
 18  activities  649 non-null    str  
 19  nursery     649 non-null    str  
 20  higher      649 non-null    str  
 21  inte

In [113]:
#identifying the target variable
#G3 is the final grade, it also is highly correlated with the other two grades G1 and G2, so we will be using it as our target variable
#i will store it under the name y_G3
y_G3 = y["G3"]
#the features remain the same, so i will keep them under the name X


In [114]:
# feature engineering and data cleaning
#checking for null values in the features dataset and the target variable
print(X.isnull().sum())
print(y_G3.isnull().sum())
#there are no null values in the features dataset, so we can proceed to the next step

school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
dtype: int64
0


In [115]:
#we will split the dataset into
#Categorical and nominal
# Then we will split the categorical features into ordinal and nominal, ordinal we will encode using label encoding and nominal we will encode using one hot encoding
categorical_features_nominal = X.select_dtypes(include=["str"]).columns
categorical_features_ordinal = X.select_dtypes(include=["int64", "float64"]).columns
print(f"Categorical features nominal: {categorical_features_nominal}")
print(f"Categorical features ordinal: {categorical_features_ordinal}")
#this matches up with what variables we need to split into categorical and numerical from the brief
# essentially categorical features have nominal data, so there is no order of significance. Ordinal data there is an order of significance ex: more absences are worse



Categorical features nominal: Index(['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
       'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic'],
      dtype='str')
Categorical features ordinal: Index(['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences'],
      dtype='str')


In [116]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_G3, test_size=0.2, random_state=42)

In [117]:
#creating a copy of the features dataset just incase
X_copy = X.copy()


In [121]:

#one hot encoding the categorical features
one_hot = OneHotEncoder(sparse_output=False, handle_unknown='ignore') 
one_hot_encoded_train = one_hot.fit_transform(X_train[categorical_features_nominal]) #only transforming the training set, we will use the same encoder to transform the test set later on
X_train_nominal = pd.DataFrame(one_hot_encoded_train, columns=one_hot.get_feature_names_out(categorical_features_nominal), index=X_train.index) #creating a new dataframe for the one hot encoded features, and keeping the same index as the original dataframe

one_hot_encoded_test = one_hot.transform(X_test[categorical_features_nominal])
X_test_nominal = pd.DataFrame(one_hot_encoded_test, columns=one_hot.get_feature_names_out(categorical_features_nominal), index=X_test.index) #creating a new dataframe for the one hot encoded features, and keeping the same index as the original dataframe
    
#add the new df in places of the old ones for both training and test sets
X_train_encoded = pd.concat(
    [X_train.drop(columns=categorical_features_nominal), X_train_nominal],
    axis=1
)

X_test_encoded = pd.concat(
    [X_test.drop(columns=categorical_features_nominal), X_test_nominal],
    axis=1
)

In [144]:
#scaling numerical inputs for knn
StandardScaler = preprocessing.StandardScaler()

X_train_knn_scaled = StandardScaler.fit_transform(X_train_encoded)
X_test_knn_scaled = StandardScaler.transform(X_test_encoded)

X_train_knn_scaled = pd.concat(
    [pd.DataFrame(X_train_knn_scaled, columns=X_train_encoded.columns, index=X_train.index)],
    axis=1
)

X_test_knn_scaled = pd.concat(
    [pd.DataFrame(X_test_knn_scaled, columns=X_test_encoded.columns, index=X_test.index)],
    axis=1
)

In [145]:
X_train_knn_scaled

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,activities_no,activities_yes,nursery_no,nursery_yes,higher_no,higher_yes,internet_no,internet_yes,romantic_no,romantic_yes
332,0.987932,-0.437418,-0.256337,-0.754310,1.352962,-0.382133,0.068136,-0.186680,-0.196270,-0.539674,...,0.979025,-0.979025,-0.500602,0.500602,-0.351250,0.351250,-0.569192,0.569192,0.759939,-0.759939
29,-0.629534,1.350140,1.591423,-0.754310,0.121054,-0.382133,0.068136,0.753970,1.501468,3.579309,...,-1.021424,1.021424,-0.500602,0.500602,-0.351250,0.351250,-0.569192,0.569192,-1.315895,1.315895
302,0.987932,0.456361,-0.256337,-0.754310,1.352962,-0.382133,1.108214,-0.186680,-1.045139,-0.539674,...,-1.021424,1.021424,1.997595,-1.997595,-0.351250,0.351250,1.756876,-1.756876,0.759939,-0.759939
286,0.179199,-0.437418,-1.180216,-0.754310,-1.110853,-0.382133,0.068136,0.753970,-1.045139,0.490072,...,0.979025,-0.979025,1.997595,-1.997595,-0.351250,0.351250,-0.569192,0.569192,0.759939,-0.759939
554,0.179199,-1.331196,-1.180216,0.555011,-1.110853,-0.382133,-0.971942,1.694619,1.501468,0.490072,...,-1.021424,1.021424,-0.500602,0.500602,2.846974,-2.846974,-0.569192,0.569192,-1.315895,1.315895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,-1.438267,1.350140,-0.256337,-0.754310,2.584869,-0.382133,-0.971942,-0.186680,-0.196270,-0.539674,...,0.979025,-0.979025,-0.500602,0.500602,-0.351250,0.351250,-0.569192,0.569192,0.759939,-0.759939
106,-1.438267,-0.437418,-0.256337,-0.754310,2.584869,-0.382133,1.108214,-2.067979,-1.045139,-0.539674,...,0.979025,-0.979025,-0.500602,0.500602,-0.351250,0.351250,-0.569192,0.569192,0.759939,-0.759939
270,-0.629534,1.350140,1.591423,-0.754310,-1.110853,-0.382133,1.108214,-0.186680,-1.045139,-0.539674,...,-1.021424,1.021424,-0.500602,0.500602,-0.351250,0.351250,-0.569192,0.569192,0.759939,-0.759939
435,-1.438267,-1.331196,-1.180216,0.555011,0.121054,-0.382133,1.108214,0.753970,-0.196270,-0.539674,...,0.979025,-0.979025,-0.500602,0.500602,-0.351250,0.351250,-0.569192,0.569192,0.759939,-0.759939
